In [4]:
import numpy as np
from numba import cuda
import math

BLOCK_SIZE = 256

@cuda.jit
def reduction_sum_kernel(d_input, d_output):
    # Allocate a shared memory array local to this specific thread block
    s_data = cuda.shared.array(shape=BLOCK_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x

    # Map global thread position to input data array
    global_idx = bx * bdim + tx

    # 1. Load data from slow global memory into fast shared memory
    if global_idx < d_input.size:
        s_data[tx] = d_input[global_idx]
    else:
        s_data[tx] = 0.0  # Pad with zero if out of bounds

    # Synchronize to guarantee all shared memory slots are filled
    cuda.syncthreads()

    # 2. Perform the in-place tree reduction inside shared memory
    stride = bdim // 2
    while stride > 0:
        if tx < stride:
            s_data[tx] += s_data[tx + stride]

        # Synchronize after every single step of the tree to avoid race conditions
        cuda.syncthreads()
        stride //= 2

    # 3. Write the block result to global memory
    # Only the first thread of the block (thread 0) writes out the total
    if tx == 0:
        d_output[bx] = s_data[0]

def parallel_reduction_sum(h_array):
    n = h_array.size
    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(n / threads_per_block)

    # Memory management: Allocation and Host-to-Device transfer
    d_input = cuda.to_device(h_array.astype(np.float32))
    d_output = cuda.device_array(blocks_per_grid, dtype=np.float32)

    # Execute the parallel kernel
    reduction_sum_kernel[blocks_per_grid, threads_per_block](d_input, d_output)

    # Bring back block sums to host and finish the last step on CPU
    h_block_sums = d_output.copy_to_host()
    return np.sum(h_block_sums)

if __name__ == "__main__":
    data = np.random.rand(500_000).astype(np.float32)

    gpu_sum = parallel_reduction_sum(data)
    cpu_sum = np.sum(data)

    print(f"Parallel Pattern GPU Sum: {gpu_sum:.4f}")
    print(f"Standard CPU Sequential Sum: {cpu_sum:.4f}")
    print(f"Patterns Match: {np.isclose(gpu_sum, cpu_sum)}")

Parallel Pattern GPU Sum: 250013.1562
Standard CPU Sequential Sum: 250013.1562
Patterns Match: True
